In [6]:
#Librerías
import pandas as pd
from pathlib import Path
import os
import re

In [7]:
from pathlib import Path
import os

# Carpeta raíz del proyecto
proyecto = Path.cwd().parent

print("Proyecto:")
print(proyecto)

Proyecto:
C:\Users\monic\Documents\CUC\Mineria de Textos\An-lisis-Sem-ntico-de-Rese-as-Tur-sticas-de-Costa-Rica-


In [8]:
#Carga del dataset
print("Directorio de trabajo actual:")
print(os.getcwd())

ruta_final = proyecto / "data" / "raw"
archivo = "reseñas_costa_rica_nuevas.csv"

df = pd.read_csv(ruta_final / archivo)
#df = pd.read_csv("../data/raw/reseñas_costa_rica_nuevas.csv")
print("Dataset cargado")
print("Dimensiones:", df.shape)
print("Columnas:", df.columns.tolist())

df.head(3)

Directorio de trabajo actual:
C:\Users\monic\Documents\CUC\Mineria de Textos\An-lisis-Sem-ntico-de-Rese-as-Tur-sticas-de-Costa-Rica-\notebooks
Dataset cargado
Dimensiones: (9948, 10)
Columnas: ['lugar', 'place_id', 'autor', 'calificacion', 'fecha_relativa', 'fecha_exacta', 'idioma_original', 'texto_original', 'fuente', 'fecha_recopilacion']


,lugar,place_id,autor,calificacion,fecha_relativa,fecha_exacta,idioma_original,texto_original,fuente,fecha_recopilacion
0,Hotel Arenal Montechiari,ChIJHaldU4oMoI8RHPCMRU6CEro,Tim Shaw,5,a month ago,2026-06-06T22:32:44.312671730Z,en,Absolutely stunning spot. The grounds are gorg...,google_places_api,2026-07-27T20:33:12.610824
1,Hotel Arenal Montechiari,ChIJHaldU4oMoI8RHPCMRU6CEro,K Love,5,7 months ago,2025-12-29T19:45:02.406352074Z,en,We stayed here for three nights with family an...,google_places_api,2026-07-27T20:33:12.617851
2,Hotel Arenal Montechiari,ChIJHaldU4oMoI8RHPCMRU6CEro,EL Broverlander,5,3 months ago,2026-04-10T17:24:14.359674268Z,en,We enjoyed a wonderful 3 bed cabin on the quie...,google_places_api,2026-07-27T20:33:12.617851


In [9]:
columnas_deseadas = [
    "texto_original",
    "calificacion",
    "fuente",
    "lugar",
    "fecha_exacta",
    "idioma_original"
]

columnas_presentes = [c for c in columnas_deseadas if c in df.columns]

df = df[columnas_presentes].copy()

print("Columnas conservadas:")
print(df.columns.tolist())

print(df.shape)

df.head()

Columnas conservadas:
['texto_original', 'calificacion', 'fuente', 'lugar', 'fecha_exacta', 'idioma_original']
(9948, 6)


,texto_original,calificacion,fuente,lugar,fecha_exacta,idioma_original
0,Absolutely stunning spot. The grounds are gorg...,5,google_places_api,Hotel Arenal Montechiari,2026-06-06T22:32:44.312671730Z,en
1,We stayed here for three nights with family an...,5,google_places_api,Hotel Arenal Montechiari,2025-12-29T19:45:02.406352074Z,en
2,We enjoyed a wonderful 3 bed cabin on the quie...,5,google_places_api,Hotel Arenal Montechiari,2026-04-10T17:24:14.359674268Z,en
3,I recently stayed at Hotel Montechiari in La F...,3,google_places_api,Hotel Arenal Montechiari,2025-11-29T02:56:35.840358616Z,en
4,We stayed two nights at Arenal Montechiari. Th...,5,google_places_api,Hotel Arenal Montechiari,2025-12-16T18:14:57.243736508Z,en


In [ ]:
duplicados = df.duplicated()

print("Duplicados encontrados:", duplicados.sum())

df = df.drop_duplicates()

print("Nuevo tamaño:", df.shape)

In [ ]:
filas_vacias = df.isnull().all(axis=1)

print("Filas completamente vacías:", filas_vacias.sum())

df = df[~filas_vacias].copy()

print(df.shape)

In [ ]:
mapeo = {
    "texto_original":"reseña",
    "calificacion":"calificación",
    "fuente":"fuente",
    "lugar":"nombre",
    "fecha_exacta":"fecha",
    "idioma_original":"idioma"
}

df.rename(columns=mapeo,inplace=True)

print(df.columns.tolist())

df.head()

In [ ]:
df = df.map(lambda x: x.lower() if isinstance(x,str) else x)

print("Texto convertido a minúsculas.")

df.head()

In [ ]:
df["fecha"] = pd.to_datetime(
    df["fecha"],
    format="mixed",
    errors="coerce"
).dt.date

df.head()

In [ ]:
condicion = df["reseña"].isna() | (df["reseña"].str.strip()=="")

print("Reseñas vacías:", condicion.sum())

df = df[~condicion].copy()

print(df.shape)

In [ ]:
def limpiar(texto):

    if not isinstance(texto,str):
        return ""

    texto = re.sub(r"[^a-zA-ZáéíóúÁÉÍÓÚñÑüÜ0-9\s]"," ",texto)

    texto = re.sub(r"\s+"," ",texto)

    return texto.strip()

df["reseña"] = df["reseña"].apply(limpiar)

df["nombre"] = df["nombre"].apply(limpiar)

print("Caracteres especiales eliminados.")

df.head()

In [ ]:
vacias = (df["reseña"]=="") | (df["reseña"].isna())

print(vacias.sum())

df = df[~vacias].copy()

print(df.shape)

In [ ]:
df_es = df[df["idioma"]=="es"].copy()

df_en = df[df["idioma"]=="en"].copy()

print("Español:",len(df_es))
print("Inglés:",len(df_en))

In [ ]:
df_en = df_en.sample(
    n=1000,
    random_state=42
)

print(df_en.shape)

In [ ]:
from deep_translator import GoogleTranslator

traductor = GoogleTranslator(source="en",target="es")

df_en["reseña"] = df_en["reseña"].apply(
    lambda x: traductor.translate(x)
) #instalar libreria de traducción

In [ ]:
df = pd.concat(
    [df_es,df_en],
    ignore_index=True
)

print(df.shape)

In [ ]:
df.drop(columns=["idioma"],inplace=True)

df.head()

In [ ]:
df["calificación"] = pd.to_numeric(
    df["calificación"],
    errors="coerce"
).astype("Int64")

df.head()

In [ ]:
ruta_salida = proyecto / "data" / "processed" / "reseñas_clean.csv"

ruta_salida.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    ruta_salida,
    index=False
)

print("Archivo guardado correctamente.")
print(df.shape)

In [ ]:
# Ruta donde se guardará el archivo
ruta_salida = proyecto / "data" / "processed" / "reseñas_clean_P2.csv"

# Crear la carpeta si no existe
ruta_salida.parent.mkdir(parents=True, exist_ok=True)

# Guardar el CSV
df.to_csv(ruta_salida, index=False)

print("Archivo guardado en:")
print(ruta_salida)